# Train the CNN guidance-map (Colab / GPU) — hard-scenario model

Consumes `guidance_dataset*.npz` shard(s) built on the planner machine via
`python -m ml_planner.build_dataset` (hard maps, GRID_RES=384).

**Hard I/O contract (must match `ml_planner/guidance.py`):** input `channels`
shape `(1,4,384,384)` float32, output `cost_to_go` shape `(1,1,384,384)`
float32, opset >= 11, names exactly `channels` / `cost_to_go`.

Design: the U-Net predicts a **residual over Euclid** (its output adds the
normalized distance-to-goal input channel), so it only learns the detour
correction Euclid misses — best chance to beat the hand-crafted heuristic.
Trained with masked MSE (labeled cells only). torch is used ONLY here.

Set **Runtime -> Change runtime type -> GPU** (T4 is enough).


In [ ]:
# 1. Load all dataset shards (upload guidance_dataset*.npz, or mount Drive).
import glob, numpy as np
try:
    from google.colab import files
    up = files.upload()                 # select one or more guidance_dataset*.npz
    paths = sorted(up.keys())
except Exception:
    paths = sorted(glob.glob('guidance_dataset*.npz'))
assert paths, 'no guidance_dataset*.npz found'
chs, las, mss, afs = [], [], [], []
for p in paths:
    d = np.load(p)
    chs.append(d['channels']); las.append(d['label']); mss.append(d['mask']); afs.append(d['affine'])
channels = np.concatenate(chs).astype('float32')   # (N,4,384,384)
label    = np.concatenate(las).astype('float32')   # (N,384,384) cost-to-go, meters
mask     = np.concatenate(mss).astype('float32')   # (N,384,384)
affine   = np.concatenate(afs)                      # (N,4): x0,y0,scale,grid_res
N, C, H, W = channels.shape
assert (C, H, W) == (4, 384, 384), f'unexpected shape {channels.shape}'
print('shards', len(paths), '| samples', N, '| grid', H, W, '| labeled cells', int(mask.sum()))


In [ ]:
# 2. Normalize cost-to-go by each sample's crop diagonal (ranking-invariant,
# stabilizes training); the residual target = normalized cost-to-go minus the
# normalized distance-to-goal channel (channel 2).
side = affine[:, 3] / affine[:, 2]                 # grid_res / scale = crop side (m)
diag = (np.sqrt(2.0) * side).astype('float32')     # (N,)
label_n = label / diag[:, None, None]              # full normalized cost-to-go
rng = np.random.default_rng(0)
idx = rng.permutation(N)
nval = max(1, N // 5)
val_idx, tr_idx = idx[:nval], idx[nval:]
print('train', len(tr_idx), 'val', len(val_idx))


In [ ]:
# 3. 3-level U-Net; output adds the dist-to-goal channel => residual-over-Euclid.
import torch, torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
            nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, cin=4, base=48):
        super().__init__()
        self.d1 = DoubleConv(cin, base);     self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(base, base*2);  self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(base*2, base*4);self.p3 = nn.MaxPool2d(2)
        self.mid = DoubleConv(base*4, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2); self.c3 = DoubleConv(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2); self.c2 = DoubleConv(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, stride=2);   self.c1 = DoubleConv(base*2, base)
        self.head = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        x1 = self.d1(x); x2 = self.d2(self.p1(x1)); x3 = self.d3(self.p2(x2)); m = self.mid(self.p3(x3))
        y = self.c3(torch.cat([self.u3(m), x3], 1))
        y = self.c2(torch.cat([self.u2(y), x2], 1))
        y = self.c1(torch.cat([self.u1(y), x1], 1))
        return self.head(y) + x[:, 2:3]     # residual over normalized dist-to-goal

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = UNet().to(dev)
print('device', dev, '| params', sum(p.numel() for p in model.parameters()))


In [ ]:
# 4. Train: masked MSE on the FULL normalized cost-to-go (the residual skip is
# inside the model), Adam + cosine LR, keep best-val weights.
Xtr = torch.tensor(channels[tr_idx]); Ytr = torch.tensor(label_n[tr_idx]); Mtr = torch.tensor(mask[tr_idx])
Xva = torch.tensor(channels[val_idx]); Yva = torch.tensor(label_n[val_idx]); Mva = torch.tensor(mask[val_idx])
EPOCHS, bs = 120, 8
opt = torch.optim.Adam(model.parameters(), 2e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

def masked_mse(pred, y, m):
    m = m > 0
    return (((pred - y) ** 2) * m).sum() / m.sum().clamp(min=1)

best, best_state = 1e9, None
for epoch in range(EPOCHS):
    model.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), bs):
        j = perm[i:i+bs]
        loss = masked_mse(model(Xtr[j].to(dev))[:, 0], Ytr[j].to(dev), Mtr[j].to(dev))
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
    model.eval()
    with torch.no_grad():
        vp = torch.cat([model(Xva[k:k+bs].to(dev))[:, 0].cpu() for k in range(0, len(Xva), bs)])
        vloss = float(masked_mse(vp, Yva, Mva))
    if vloss < best:
        best = vloss; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if epoch % 10 == 0:
        print(f'epoch {epoch:3d}  val masked-MSE {vloss:.6f}')
print('best val masked-MSE', best)
model.load_state_dict(best_state)


In [ ]:
# 5. Export a SINGLE self-contained ONNX (channels -> cost_to_go, 384x384) and download.
model.eval().cpu()
dummy = torch.zeros(1, 4, 384, 384)
torch.onnx.export(
    model, dummy, 'guidance.onnx',
    input_names=['channels'], output_names=['cost_to_go'],
    opset_version=13,
    dynamic_axes={'channels': {0: 'batch'}, 'cost_to_go': {0: 'batch'}})
# Force one file (some torch/onnx versions emit an external weights sidecar).
import onnx
onnx.save_model(onnx.load('guidance.onnx'), 'guidance.onnx', save_as_external_data=False)
print('exported single-file guidance.onnx')
try:
    from google.colab import files
    files.download('guidance.onnx')
except Exception:
    pass
# Drop guidance.onnx into ml_planner/models/ on the planner machine; then
# secondary='guidance' activates automatically (falls back to hand-crafted if absent).
